In [1]:
import sys
import os
import time
from lightkurve import search_targetpixelfile
from lightkurve import TessTargetPixelFile
import lightkurve as lk
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from sklearn.model_selection import train_test_split
from sklearn.tree import DecisionTreeClassifier
from keras.models import load_model
from keras.optimizers import Adam
import tensorflow as tf
from tensorflow.keras.models import Sequential
from tensorflow.keras.layers import Dense, Dropout, Activation, Conv1D, MaxPooling1D, Flatten
from wotan import slide_clip
from wotan import transit_mask, flatten
from astropy.stats import sigma_clip
from astropy import units as u
import csv
import shutil
from scipy.interpolate import interp1d
from tsfresh import extract_features
from tsfresh.utilities.dataframe_functions import make_forecasting_frame
from tsfresh.utilities.dataframe_functions import impute
from statsmodels.tsa.seasonal import seasonal_decompose
from multiprocessing import Pool
import multiprocessing
import numpy as np
import pandas as pd
import lightkurve as lk
from scipy.signal import find_peaks
from astropy.timeseries import BoxLeastSquares
from scipy.interpolate import interp1d
from tslearn.preprocessing import TimeSeriesResampler
from sklearn.preprocessing import MinMaxScaler

In [2]:
%matplotlib inline

In [ ]:
value_df = 10000
star_check = pd.read_csv("./Datasets/updated_database_exoplanet.csv")

In [2]:
def get_planet_ids():
    stars1 = pd.read_csv('./Datasets/star1.csv')
    stars2 = pd.read_csv('./Datasets/star2.csv')
    stars3 = pd.read_csv('./Datasets/star3.csv')
    stars4 = pd.read_csv('./Datasets/star4.csv')
    stars = pd.concat([stars1,stars2,stars3,stars4])
    stars = stars.drop_duplicates('target_name')
    stars['target_name'] = stars['target_name'].apply(lambda x: 'TIC ' + str(x))
    confirmed_stars = pd.read_csv('./Datasets/confirmed_stars.csv')
    confirmed_stars['tid'] = confirmed_stars['tid'].apply(lambda x: 'TIC ' + str(x))
    stars['confirmed_planet'] = stars['target_name'].isin(confirmed_stars['tid']).astype(int)
    stars = stars.reset_index()
    confirmed_index = np.array(stars[stars['confirmed_planet']==1].index)
    unconfirmed_index = np.array(stars[stars['confirmed_planet']==0].index)
    return stars, confirmed_index, unconfirmed_index

In [3]:
t1, t2, t3 = get_planet_ids()

/var/folders/f6/5xg9w0c9187d11pcnd4qcqlm0000gn/T/ipykernel_94929/3048257929.py:3: DtypeWarning: Columns (4) have mixed types. Specify dtype option on import or set low_memory=False.
  stars2 = pd.read_csv('./Datasets/star2.csv')
/var/folders/f6/5xg9w0c9187d11pcnd4qcqlm0000gn/T/ipykernel_94929/3048257929.py:5: DtypeWarning: Columns (4) have mixed types. Specify dtype option on import or set low_memory=False.
  stars4 = pd.read_csv('./Datasets/star4.csv')


In [ ]:
len(t3)

In [11]:
[t1['target_name'][i] for i in t2].index('TIC 261264800')

1854

In [5]:
[t1['target_name'][i] for i in t3].index('TIC 233208532')

6642

In [ ]:
len(t2)

In [ ]:
4692,5163
6206,6780

In [ ]:
star_check

In [12]:
from astroquery.mast import Catalogs
catalog_data = Catalogs.query_criteria(catalog="Tic", ID='282923567')

In [14]:
catalog_data = catalog_data.to_pandas()

In [15]:
catalog_data.to_json()

'{"ID":{"0":"282923567"},"version":{"0":"20190415"},"HIP":{"0":null},"TYC":{"0":"4183-00484-1"},"UCAC":{"0":"765-046512"},"TWOMASS":{"0":"14544136+6248588"},"SDSS":{"0":"1237654949982240801"},"ALLWISE":{"0":"J145441.36+624858.6"},"GAIA":{"0":"1620312288278786688"},"APASS":{"0":"58625762"},"KIC":{"0":null},"objType":{"0":"STAR"},"typeSrc":{"0":"tmgaia2"},"ra":{"0":223.6723394966},"dec":{"0":62.8163079568},"POSflag":{"0":"tmgaia2"},"pmRA":{"0":2.1411},"e_pmRA":{"0":0.0508882},"pmDEC":{"0":-4.1457},"e_pmDEC":{"0":0.0493099},"PMflag":{"0":"gaia2"},"plx":{"0":1.01639},"e_plx":{"0":0.0267637},"PARflag":{"0":"gaia2"},"gallong":{"0":102.0306518137},"gallat":{"0":49.0090705651},"eclong":{"0":168.9706035764},"eclat":{"0":70.3268347576},"Bmag":{"0":10.957},"e_Bmag":{"0":0.049},"Vmag":{"0":9.46},"e_Vmag":{"0":0.009},"umag":{"0":15.2018},"e_umag":{"0":0.00820231},"gmag":{"0":10.4066},"e_gmag":{"0":0.000345804},"rmag":{"0":9.15463},"e_rmag":{"0":0.000156558},"imag":{"0":8.82452},"e_imag":{"0":0.0003

In [5]:
sectorsdata = lk.search_lightcurve('TIC 282923567', author=["TESS-SPOC"], exptime=1800)

In [6]:
search_data = sectorsdata.table.to_pandas(index=False)

In [10]:
search_data.to_json()

'{"intentType":{"0":"science","1":"science","2":"science","3":"science","4":"science"},"obs_collection":{"0":"HLSP","1":"HLSP","2":"HLSP","3":"HLSP","4":"HLSP"},"provenance_name":{"0":"TESS-SPOC","1":"TESS-SPOC","2":"TESS-SPOC","3":"TESS-SPOC","4":"TESS-SPOC"},"instrument_name":{"0":"Photometer","1":"Photometer","2":"Photometer","3":"Photometer","4":"Photometer"},"project":{"0":"TESS","1":"TESS","2":"TESS","3":"TESS","4":"TESS"},"filters":{"0":"TESS","1":"TESS","2":"TESS","3":"TESS","4":"TESS"},"wavelength_region":{"0":"Optical","1":"Optical","2":"Optical","3":"Optical","4":"Optical"},"target_name":{"0":"282923567","1":"282923567","2":"282923567","3":"282923567","4":"282923567"},"target_classification":{"0":null,"1":null,"2":null,"3":null,"4":null},"obs_id":{"0":"hlsp_tess-spoc_tess_phot_0000000282923567-s0015_tess_v1_tp","1":"hlsp_tess-spoc_tess_phot_0000000282923567-s0016_tess_v1_tp","2":"hlsp_tess-spoc_tess_phot_0000000282923567-s0021_tess_v1_tp","3":"hlsp_tess-spoc_tess_phot_000000

In [28]:
search_data

,intentType,obs_collection,provenance_name,instrument_name,project,filters,wavelength_region,target_name,target_classification,obs_id,...,productFilename,size,parent_obsid,dataRights_products,calib_level_products,author,mission,#,year,sort_order
0,science,HLSP,TESS-SPOC,Photometer,TESS,TESS,Optical,282923567,NaN,hlsp_tess-spoc_tess_phot_0000000282923567-s001...,...,hlsp_tess-spoc_tess_phot_0000000282923567-s001...,155520,41903037,PUBLIC,4,TESS-SPOC,TESS Sector 15,0,2019,2
1,science,HLSP,TESS-SPOC,Photometer,TESS,TESS,Optical,282923567,NaN,hlsp_tess-spoc_tess_phot_0000000282923567-s001...,...,hlsp_tess-spoc_tess_phot_0000000282923567-s001...,149760,40703228,PUBLIC,4,TESS-SPOC,TESS Sector 16,1,2019,2
2,science,HLSP,TESS-SPOC,Photometer,TESS,TESS,Optical,282923567,NaN,hlsp_tess-spoc_tess_phot_0000000282923567-s002...,...,hlsp_tess-spoc_tess_phot_0000000282923567-s002...,161280,36205077,PUBLIC,4,TESS-SPOC,TESS Sector 21,2,2020,2
3,science,HLSP,TESS-SPOC,Photometer,TESS,TESS,Optical,282923567,NaN,hlsp_tess-spoc_tess_phot_0000000282923567-s002...,...,hlsp_tess-spoc_tess_phot_0000000282923567-s002...,161280,35698120,PUBLIC,4,TESS-SPOC,TESS Sector 22,3,2020,2
4,science,HLSP,TESS-SPOC,Photometer,TESS,TESS,Optical,282923567,NaN,hlsp_tess-spoc_tess_phot_0000000282923567-s002...,...,hlsp_tess-spoc_tess_phot_0000000282923567-s002...,158400,35281893,PUBLIC,4,TESS-SPOC,TESS Sector 23,4,2020,2


In [22]:
for i in search_data.columns:
    print(i)

intentType
obs_collection
provenance_name
instrument_name
project
filters
wavelength_region
target_name
target_classification
obs_id
s_ra
s_dec
dataproduct_type
proposal_pi
calib_level
t_min
t_max
t_exptime
em_min
em_max
obs_title
t_obs_release
proposal_id
proposal_type
sequence_number
s_region
jpegURL
dataURL
dataRights
mtFlag
srcDen
obsid
objID
exptime
distance
obsID
obs_collection_products
dataproduct_type_products
description
type
dataURI
productType
productGroupDescription
productSubGroupDescription
productDocumentationURL
project_products
prvversion
proposal_id_products
productFilename
size
parent_obsid
dataRights_products
calib_level_products
author
mission
#
year
sort_order


In [ ]:
for i in catalog_data.to_pandas().columns:
    print(i)

In [7]:
catalog = {
'Unique identifier': catalog_data['ID'].values[0],
'Type of object': catalog_data['objType'].values[0],
'Source of object type': catalog_data['typeSrc'].values[0],
'Right Ascension': catalog_data['ra'].values[0],
'Declination': catalog_data['dec'].values[0],
'Parallax measurement': catalog_data['plx'].values[0],
'Nearest neighbor distance': catalog_data['prox'].values[0],
'TESS band magnitude': catalog_data['Tmag'].values[0],
'Surface temperature': catalog_data['Teff'].values[0],
'Surface gravity': catalog_data['logg'].values[0],
'Metal content': catalog_data['MH'].values[0],
'Stellar radius': catalog_data['rad'].values[0],
'Stellar mass': catalog_data['mass'].values[0],
'Stellar density': catalog_data['rho'].values[0],
'Stellar luminosity': catalog_data['lum'].values[0],
'Distance to star': catalog_data['d'].values[0]
}

In [8]:
catalog

{'Unique identifier': '282923567',
 'Type of object': 'STAR',
 'Source of object type': 'tmgaia2',
 'Right Ascension': 223.672339496594,
 'Declination': 62.8163079567987,
 'Parallax measurement': 1.01639,
 'Nearest neighbor distance': nan,
 'TESS band magnitude': 8.4635,
 'Surface temperature': 4494.0,
 'Surface gravity': nan,
 'Metal content': nan,
 'Stellar radius': 23.325,
 'Stellar mass': nan,
 'Stellar density': nan,
 'Luminosity classification': 'GIANT',
 'Stellar luminosity': nan,
 'Distance to star': 956.405}

In [ ]:
sectorsdata = lk.search_lightcurve(idno, author=["TESS-SPOC"], exptime=1800)

In [ ]:
def smn(i):
    idno = star_check[star_check['confirmed_planet']==1]['tid'].iloc[4]
    sectorsdata = lk.search_lightcurve(idno, author=["TESS-SPOC"], exptime=1800)
    return len(sectorsdata)

t = [smn(i) for i in range(0,50)]

In [ ]:
t

In [ ]:
idno = star_check[star_check['confirmed_planet']==1]['tid'].iloc[4]
sectorsdata = lk.search_lightcurve(idno, author=["TESS-SPOC"], exptime=1800)
if(sectorsdata.download_all()!= None):
    sectors = sectorsdata.download_all()
    shutil.rmtree('/Users/nasvinnabeel/.lightkurve/cache/mastDownload/')
    ax = sectors.stitch().plot(color='#1e77b4')
    ax.legend().set_visible(False)
    plt.show()

In [ ]:
idno = star_check[star_check['confirmed_planet']==1]['tid'].iloc[4]
sectorsdata = lk.search_lightcurve(idno, author=["TESS-SPOC"], exptime=1800)
if(sectorsdata.download_all()!= None):
    sectors = sectorsdata.download_all()
    shutil.rmtree('/Users/nasvinnabeel/.lightkurve/cache/mastDownload/')
    ax = sectors.stitch().flatten().plot(color='#1e77b4')
    ax.legend().set_visible(False)
    plt.show()

In [ ]:
idno = star_check[star_check['confirmed_planet']==1]['tid'].iloc[3]
sectorsdata = lk.search_lightcurve(idno, author=["TESS-SPOC"], exptime=1800)
if(sectorsdata.download_all()!= None):
    sectors = sectorsdata.download_all()
    shutil.rmtree('/Users/nasvinnabeel/.lightkurve/cache/mastDownload/')
    # sectors.plot()
    for i in sectors:
        i.flux = i.pdcsap_flux.value.unmasked
        i.flux_err = i.pdcsap_flux_err.value.unmasked
    stitched_lc = sectors.stitch()
    time = stitched_lc.time.value
    flux = stitched_lc.flux.value
    min_period = 1
    max_period = 1000
    num_periods = 10000
    period_time = np.logspace(np.log10(min_period), np.log10(max_period), num_periods)
    bls_periodogram = stitched_lc.to_periodogram(method='bls', period=period_time)
    planet_period = bls_periodogram.period_at_max_power
    planet_t0 = bls_periodogram.transit_time_at_max_power
    planet_duration = bls_periodogram.duration_at_max_power
    folded_light_curve = stitched_lc.fold(period=planet_period, epoch_time=planet_t0)
    # folded_light_curve.plot()
    flatten_lc, trend_lc = flatten(folded_light_curve.time.value, folded_light_curve.flux, method='biweight', return_trend=True)
    light = pd.DataFrame({'Time':folded_light_curve.time.value,'Flux':flatten_lc}).dropna()
    flux_series = pd.Series([i[0] for i in light[['Flux']].to_numpy()], index=[i[0] for i in light[['Time']].to_numpy()])
    decompose_result_mult = seasonal_decompose(flux_series, model="additive", period=int(planet_period.value))
    trend = decompose_result_mult.trend
    new_ts = TimeSeriesResampler(sz=value_df).fit_transform(trend.to_numpy())
    scaler = MinMaxScaler()
    new_ts = np.array(new_ts[0])
    new_new_ts = scaler.fit_transform(new_ts)
    t = [i[0] for i in new_new_ts]
    mean = np.nanmean(t)
    final_ts = np.nan_to_num(t, copy=False, nan=mean)
    plt.figure()
    plt.plot(final_ts)

In [ ]:
np.concatenate([[idno, 0],final_ts])

In [ ]:
scaler = MinMaxScaler()
new_ts = np.array(new_ts[0])
new_new_ts = scaler.fit_transform(new_ts)

In [ ]:
new_new_ts[np.isnan(new_new_ts)] = new_new_ts.mean()

In [ ]:
[i[0] for i in new_new_ts]

In [ ]:
plt.plot(new_new_ts)

In [ ]:
X_std = (new_ts - new_ts.min()) / (new_ts.max() - new_ts.min())
X_scaled = X_std * (1 - 0) + 1

In [ ]:
plt.plot(X_scaled)

In [ ]:
plt.plot(new_ts[0])

In [ ]:
trend.plot()

In [ ]:
plt.plot(new_ts[0])

In [ ]:
trend.plot()

In [ ]:
star_check.iloc[900].drop(['tid','confirmed_planet','Teff','logg','MH','rad','mass','rho','lum','Tmag','ra','dec','plx']).plot()

In [16]:
new_data_1 = pd.read_csv('./Datasets/new_data_1.csv')
new_data_2 = pd.read_csv('./Datasets/new_data_2.csv')

In [31]:
star_list = [int(i[0][4:]) for i in new_data_1[['tid']].to_numpy()]
catalog_data = Catalogs.query_criteria(catalog="Tic", ID=star_list)
catalog_data = catalog_data[['ID','Teff','logg','MH','rad','mass','rho','lum','Tmag','ra','dec','plx']]
catalog_data['ID'] = ['TIC ' + str(id) for id in catalog_data['ID']]
catalog_data = catalog_data.to_pandas()
catalog_data = catalog_data.rename(columns={'ID': 'tid'})
new_star_1 = pd.merge(new_data_1, catalog_data, on='tid', how='inner')
# new_star_1 = new_star_1.drop_duplicates(subset='tid', keep='first')

In [32]:
star_list = [int(i[0][4:]) for i in new_data_2[['tid']].to_numpy()]
catalog_data = Catalogs.query_criteria(catalog="Tic", ID=star_list)
catalog_data = catalog_data[['ID','Teff','logg','MH','rad','mass','rho','lum','Tmag','ra','dec','plx']]
catalog_data['ID'] = ['TIC ' + str(id) for id in catalog_data['ID']]
catalog_data = catalog_data.to_pandas()
catalog_data = catalog_data.rename(columns={'ID': 'tid'})
new_star_2 = pd.merge(new_data_2, catalog_data, on='tid', how='inner')
# new_star_2 = new_star_2.drop_duplicates(subset='tid', keep='first')

In [35]:
new_data = pd.concat([new_star_1,new_star_2])
# new_data = new_data.drop_duplicates(subset='tid', keep='first')

In [36]:
new_data

,tid,confirmed_planet,1,2,3,4,5,6,7,8,...,logg,MH,rad,mass,rho,lum,Tmag,ra,dec,plx
0,TIC 18067025,1,0.833303,0.833303,0.833303,0.833303,0.833303,0.833303,0.833303,0.833303,...,4.52282,NaN,0.795922,0.770000,1.527140,0.296211,13.08740,184.759983,36.609666,3.11271
1,TIC 4749723,0,0.515608,0.515608,0.515608,0.515608,0.515608,0.515608,0.515608,0.515608,...,3.57778,-0.0290,2.670800,0.984000,0.051650,6.216298,8.14936,188.826441,33.047742,6.72848
2,TIC 23769326,1,0.805253,0.805253,0.805253,0.805253,0.805253,0.805253,0.805253,0.805253,...,4.27706,NaN,1.318540,1.200000,0.523483,2.355028,12.33420,208.944741,40.391802,1.62523
3,TIC 4749946,0,0.481566,0.481566,0.481566,0.481566,0.481566,0.481566,0.481566,0.481566,...,4.59371,0.2135,0.766162,0.840000,1.867750,0.338409,9.75140,188.904944,35.027144,13.32070
4,TIC 139086171,1,0.705327,0.705327,0.705327,0.705327,0.705327,0.705327,0.705327,0.705327,...,4.47278,-0.0940,0.921586,0.920000,1.175380,0.618640,11.30930,184.625038,34.898123,4.95353
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
2411,TIC 233207585,0,0.294832,0.235996,0.324474,0.218544,0.276013,0.252484,0.276107,0.227149,...,4.32473,NaN,1.100550,0.933000,0.699924,0.914887,10.20130,266.001392,70.258838,6.68597
2412,TIC 155001079,1,0.682971,0.720092,0.757214,0.794336,0.831457,0.868579,0.904268,0.906554,...,3.64947,NaN,3.838240,2.397000,0.042391,104.999992,9.06657,345.155438,46.032551,1.65308
2413,TIC 233208400,0,0.681390,0.681390,0.681390,0.681390,0.681390,0.681390,0.681390,0.681390,...,4.70047,NaN,0.542438,0.538402,3.373310,0.045272,12.63500,266.336830,70.901541,9.83047
2414,TIC 364074068,1,0.636959,0.636959,0.636959,0.636959,0.636959,0.636959,0.636959,0.636959,...,4.68309,NaN,0.562607,0.556454,3.124740,0.056755,12.09480,306.329027,77.636188,11.50290


In [37]:
new_data.value_counts('confirmed_planet')

confirmed_planet
0    5905
1    4183
Name: count, dtype: int64

In [38]:
new_data.to_csv('./Datasets/final_dataset.csv', index=False)